# AI-Generated Exploit Testbenches (with transcripts)

## 1. Setup

In [1]:
from pathlib import Path
import os, sys

def find_repo_root(markers, g):
    starts = []
    for var in ("__vsc_ipynb_file__", "__file__", "__session__"):
        val = g.get(var)
        if val:
            try:
                starts.append(Path(val).resolve().parent)
            except Exception:
                pass
    starts.append(Path.cwd().resolve())
    for start in starts:
        for d in (start, *start.parents):
            for sub in ("", "aha2026-dp1"):
                cand = d / sub if sub else d
                if all((cand / m).exists() for m in markers):
                    return cand
    raise FileNotFoundError(f"Open this notebook inside the aha2026-dp1 repository (cwd={Path.cwd()})")

ROOT = find_repo_root(("recovery/recovered_sim.v", "tb/trojan_pipeline"), globals())
if str(ROOT / "tb") not in sys.path:
    sys.path.insert(0, str(ROOT / "tb"))

from trojan_pipeline.exploit import generate_and_verify, VARIANTS, SPECS, DEFAULT_TB_MODEL
from spi_workflow.common import load_portkey_key
from portkey_ai import Portkey

MODEL = os.getenv("PORTKEY_MODEL", DEFAULT_TB_MODEL)
API_KEY = os.environ.get("PORTKEY_API_KEY") or load_portkey_key(ROOT)
client = Portkey(api_key=API_KEY,
                 base_url=os.getenv("PORTKEY_BASE_URL", "https://ai-gateway.apps.cloud.rt.nyu.edu/v1"),
                 max_retries=0)
TBDIR = ROOT / "tb/trojan_verification"
ORDER = ["baseline", "functionality", "dos", "performance"]

print("Repository :", ROOT)
print("Model      :", MODEL)
print("Key loaded :", bool(API_KEY))
print("Targets    :", ORDER)

Repository : /home/bluto/Desktop/_ece4313/AHA/aha2026-dp1
Model      : @vertexai/gemini-3.1-pro-preview
Key loaded : True
Targets    : ['baseline', 'functionality', 'dos', 'performance']


## 2. Generate and verify each exploit testbench

In [2]:
RESULTS = {}
for variant in ORDER:
    print(f"\n=== {variant} ===  ({SPECS[variant][:70]}...)")
    code_sv, sim_log, run = generate_and_verify(variant, client, ROOT, model=MODEL, max_attempts=5)
    dest = TBDIR / f"{variant}_tb.sv"
    dest.write_text(code_sv)            # AI-authored testbench replaces the file
    attempts = len(list((ROOT / "ai_logs/exploit_tb" / variant).glob("attempt_*")))
    summary = next((l for l in sim_log.splitlines() if "SUMMARY" in l), "")
    print(f"  AI-generated -> {dest.relative_to(ROOT)}  ({attempts} attempt(s), {len(code_sv)} chars)")
    print(f"  transcript   -> {run.relative_to(ROOT)}")
    print(f"  {summary}")
    RESULTS[variant] = {"file": dest, "transcript": run, "attempts": attempts}


=== baseline ===  (Prove normal operation of the recovered accelerator: a full encryption...)
  AI-generated -> tb/trojan_verification/baseline_tb.sv  (1 attempt(s), 1945 chars)
  transcript   -> ai_logs/exploit_tb/baseline/attempt_001
  BASELINE SUMMARY PASS - Checks: 8, Total Pulses: 148

=== functionality ===  (Functionality-corruption Trojan. Trigger: a complete 32-bit SPI transa...)
  AI-generated -> tb/trojan_verification/functionality_tb.sv  (3 attempt(s), 4174 chars)
  transcript   -> ai_logs/exploit_tb/functionality/attempt_003
  FUNCTIONALITY SUMMARY PASS - All 16 checks completed successfully in 408 clock pulses.

=== dos ===  (Denial-of-service Trojan. Trigger: a complete 32-bit SPI transaction w...)
  AI-generated -> tb/trojan_verification/dos_tb.sv  (1 attempt(s), 4187 chars)
  transcript   -> ai_logs/exploit_tb/dos/attempt_001
  DOS SUMMARY PASS: All tests passed successfully.

=== performance ===  (Performance-degradation Trojan. Trigger: the 4th START operation after.

## 3. Provenance summary

In [3]:
print("AI-authored exploit testbenches (model: %s):" % MODEL)
print("=" * 60)
for variant in ORDER:
    r = RESULTS[variant]
    print(f"  {str(r['file'].relative_to(ROOT)):40s}  <-  {r['transcript'].parent.relative_to(ROOT)}/  ({r['attempts']} attempt(s))")
print("=" * 60)
print("Each testbench was generated by the model and accepted only after its")
print("simulation against the actual Trojan printed SUMMARY PASS. Prompts and")
print("responses are under ai_logs/exploit_tb/. The reused spi_operations.svh and")
print("crypto_dut.sv are AI-generated by spi_tb_gen (ai_logs/tb/).")
print("\nRe-verify them any time with notebooks/trojan_verification.ipynb.")

AI-authored exploit testbenches (model: @vertexai/gemini-3.1-pro-preview):
  tb/trojan_verification/baseline_tb.sv     <-  ai_logs/exploit_tb/baseline/  (1 attempt(s))
  tb/trojan_verification/functionality_tb.sv  <-  ai_logs/exploit_tb/functionality/  (3 attempt(s))
  tb/trojan_verification/dos_tb.sv          <-  ai_logs/exploit_tb/dos/  (1 attempt(s))
  tb/trojan_verification/performance_tb.sv  <-  ai_logs/exploit_tb/performance/  (2 attempt(s))
Each testbench was generated by the model and accepted only after its
simulation against the actual Trojan printed SUMMARY PASS. Prompts and
responses are under ai_logs/exploit_tb/. The reused spi_operations.svh and
crypto_dut.sv are AI-generated by spi_tb_gen (ai_logs/tb/).

Re-verify them any time with notebooks/trojan_verification.ipynb.
